# Experiment 3: representation and one-step dynamics
Uses completed **Experiment 1 recordings only**. No simulator, optimizer, training runner, or new data collection. Common-transition and native-rollout tables are separate; evaluation seeds are not independently trained policies.

In [ ]:
import json, os, sys
from pathlib import Path
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks/exp3_representation.py").exists())
sys.path.insert(0, str(ROOT / "notebooks"))
import exp3_representation as analysis

## Configuration
Completed Exp1 recordings under `recording_roots` are searched recursively for all available methods. Set `tasks` to a list of task IDs to select a subset, or set `methods` to an explicit mapping for checkpoint relocation/separate checkpoint cohorts (leave `tasks=None`). Exp2 and incomplete runs are excluded.
Use trusted checkpoints only. An omitted checkpoint uses the path recorded in metadata; an explicit path must have the same SHA256. Discovery never substitutes the latest trained model. Multiple checkpoints for one method require explicit selection/separate IDs. Sampling is deterministic, balanced over available condition/motion/phase/seed/source strata **separately per loading range**. Start with a small budget. Strict protocol checks may require selecting narrower roots.

Fresh frozen inference is the default. CPU/GPU numerical precision may differ: native parity must pass before cross-method replay. Do not increase tolerances simply to suppress a mismatch. `recorded_native_only=True` explicitly audits saved predictions instead, with no parity claim or cross-method replay.

In [ ]:
from b1z1_paper_names import DISPLAY_NAMES, LATEX_METHOD_NAMES
# Paper labels differ from implementation ablation numbers; task IDs stay unchanged.
CONFIG = {
    "recording_roots": [str(ROOT / "evaluation/exp1")],
    "tasks": None,  # All discovered tasks, or e.g. ["b1z1_pact", "b1z1_ppo_pos"].
    "methods": None,  # Optional explicit {ID: {"task": ..., "checkpoint": ...}}.
    "evaluation_seeds": [0, 1, 2],
    "conditions": ["nominal", "ee-disturbance", "combined-disturbance"],
    "sampling_budget": 256,  # Total selected transitions PER loading range.
    "sampling_seed": 0,
    "device": "cuda:0",
    "matmul_precision": "high",  # Matches the inspected Isaac Lab recording; parity still required.
    "output_dir": str(ROOT / "analysis/exp3"),
    "dynamics": True,  # Direct BARD; requires CUDA, bard-pytorch-dynamics and Pinocchio.
    "urdf": None,  # Optional local relocation; must match recorded URDF hash.
    "recorded_native_only": False,
    "parity_atol": 2e-5, "parity_rtol": 2e-4,
    "latex_method_names": {},  # Optional overrides keyed by method ID; defaults resolved below.
}
if os.environ.get("EXP3_CONFIG"):
    CONFIG.update(json.loads(os.environ["EXP3_CONFIG"]))

CONFIG = analysis.resolve_methods(CONFIG)
display(pd.DataFrame.from_dict(CONFIG["methods"], orient="index"))
# Resolve paper labels/macros even when methods were supplied by JSON overrides.
for method in CONFIG["methods"].values():
    if method["task"] in DISPLAY_NAMES:
        method["display_name"] = DISPLAY_NAMES[method["task"]]
CONFIG["latex_method_names"] = {
    **{key: LATEX_METHOD_NAMES[method["task"]] for key, method in CONFIG["methods"].items()
       if method["task"] in LATEX_METHOD_NAMES},
    **CONFIG.get("latex_method_names", {}),
}


## Execute and audit
PACT/current Pos histories share widths but not torque-channel semantics. UniFP histories additionally contain policy-specific compensation commands. The helper never pads channels or constructs force commands from privileged loads. Only identical audited input contracts can share transitions; unsupported pairs remain native-rollout comparisons.

Current force heads are scored against the **previous applied interval load visible in history_t**. Newly imposed loads are labelled separately. GRFs use the recorded source torque-conditioning input and successor-yaw conversion, and are compared with raw interval-average world forces, not processed training labels.

In [ ]:
result = analysis.run_analysis(CONFIG)
display(pd.DataFrame(result["table_summary"]).query("axis == 'all'"))
print("\n".join(result["diagnostics"]) or "No coverage diagnostics.")
display(pd.DataFrame(result["unavailable"]))

## Portable results
The page-wide tables contain EE force, torso force, torso moment, GRF, base-linear velocity, base-angular velocity, and joint-velocity component RMSE. Biases, individual axes/feet, arm/leg velocities, event phases and motion modes remain in CSV/JSON breakdowns. Table means first reduce within trials, then equally across evaluation seeds; SD uses ddof=1. Missing heads/dynamics are dashes, never zeros.

BARD uses q_t mechanics, actual interval-average executed torque, and frozen decoded loads for an analytic one-step velocity prediction. Measured-loading and constant-velocity references are separate variants. Native UniFP has no complete learned-loading dynamics result because it has no moment or GRF head. Physics losses being disabled in training does not disable these offline diagnostics.

In [ ]:
display(pd.DataFrame(result["per_seed"]).head(20))
display(pd.DataFrame(result["native_parity"].items(), columns=["Recording", "Native parity"]))
print("Manifest, numeric summaries, caches and separate LaTeX tables:", CONFIG["output_dir"])